In [8]:
# Run with a Sagemath Kernel

In [9]:
reset()

In [10]:
%run ntru.ipynb
import subprocess, re
import numpy as np
from pathlib import Path

In [11]:
flatter_path = 'flatter' ## Don't forget to add your flatter path

In [12]:
def ntruhps(x):
    '''
    Returns the parameters of NTRU N,q,d, as well as our choice of N1,N2 and k
    :param x: The NTRU variant, 1 for ntruhps2048509, 2 for ntruhps2048677, 3 for ntruhps4096821
    '''
    if x == 1: # ntruhps2048509
        N = 509; q = 2048; d = q/16-1; k = 425; N1 = 9; N2 = 8
    elif x == 2: # ntruhps2048677
        N = 677; q = 2048; d = q/16-1; k = 600; N1 = 1; N2 = 15
    elif x == 3: # ntruhps4096821
        N = 821; q = 4096; d = q/16-1; k = 760; N1 = 7; N2 = 21
    else:
        return None
    return N, q, d, k, N1, N2

def centered_mod(arr, q):
    arr = np.array(arr)
    return ((arr + q // 2) % q) - q // 2

def calculate_c_minus_m(ciphertext, known_msg):
    '''
    :param ciphertext: the ciphertext
    :param known_msg: the known coefficients of the message
    :return: a list of the differences 3^-1(c-m)%q
    '''
    output = []
    for i in range(len(known_msg)): # we know only the first 3 coefficients of the message
        three_inverse = mod(3^(-1), q)
        output.append(int(three_inverse*(ciphertext[i] - known_msg[i])))
    return output

def get_coeffs_of_right_side(N, h, k):
    return [ h[(k-i)%N] for i in range(N) ]

def create_fplll_file(filename, n, a, a0, N1, N2, q, k):
    # Create the Bk matrix
    equations = len(a0)

    zerom = matrix(1, n, 0)
    M = identity_matrix(ZZ, n)
    M = M.stack(zerom)
    lowerM = [vector(a[i])*N2 for i in range(equations)]

    for i in range(equations):
        M = M.stack(lowerM[i])

    lastCol = [0]*n + [N1]
    for i in range(equations):
        lastCol.append(-N2*a0[i])
    lactCol = vector(QQ, lastCol)
    lactCol = lactCol.column()
    M = M.augment(lactCol)
    Mt = M.transpose() # write to file and read from fpylll

    qIN = identity_matrix(ZZ, k)*q*N2
    zerom2 = matrix(k, n+1, 0)
    qIN = zerom2.augment(qIN)
    E = Mt.stack(qIN)

    f = open("matrices/"+filename, 'w')
    f.write("[")
    for i in E:
        f.write("[")
        [f.write(str(j)+" ") for j in list(i)]
        f.write("]\n")
    f.write("]\n")
    f.close()
    return E

def check_result(result):
    norm_r = vector(r[0]).norm().n()
    norm_result = vector(result).norm().n()
    # print("Norms of r and result: ", norm_r, norm_result)
    # if approx_value[variant] -3 <= norm_result <= approx_value[variant] +3:
    if norm_r == norm_result:
        print("Found same norm vector!!!!!!!!!")
        if r[0] == list(result):
            show("Correct r found: ", r[0] == list(result))
            return True
        else:
            return False
    else:
        return False

def try_row(possible_solution, homoge):
    if check_result(possible_solution):
        return True
    result_minus_one = list(vector(possible_solution)-vector([1]*N)) # we use the vector from the reduction minus a [1]*N vector
    if check_result(result_minus_one):
        return True
    result_plus_one = list(vector(possible_solution)+vector([1]*N)) # we use the vector from the reduction plus a [1]*N vector
    if check_result(result_plus_one):
        return True
    result_center =  centered_mod(possible_solution, q) # centerlift the reduction vector
    if check_result(result_center):
        return True
    result_minus = centered_mod( list(vector(possible_solution)-vector([1]*N)) , q) # centerlifted vector minus [1]*N vector
    if check_result(result_minus):
        return True
    result_plus = centered_mod( list(vector(possible_solution)+vector([1]*N)) , q) # centerlifted vector plus [1]*N vector
    if check_result(result_plus):
        return True
    result_minus_center = centered_mod( list(vector(result_center)-vector([1]*N)) , q) # centerlifted vector minus [1]*N vector
    if check_result(result_minus_center):
        return True
    result_plus_center = centered_mod( list(vector(result_center)+vector([1]*N)) , q) # centerlifted vector plus [1]*N vector
    if check_result(result_plus_center):
        return True

    for i in homoge:
        plus = possible_solution + i # add the homogeneous vector to our possible solution
        minus = possible_solution - i # subtract the homogeneous vector to our possible solution
        if check_result(plus):
            return True
        if check_result(minus):
            return True
    return False

def create_NTRU(N, q, d, k):
    Zx.<x> = ZZ[]
    D = x^N - 1

    R.<xN> = Zx.quotient(D) # ring in Zx/DN

    F2 = GF(2)
    F2x.<x2> = F2[]
    S2.<X2> = F2x.quotient(D) # ring in 2/DN

    F3 = GF(3)
    F3x.<x3> = F3[]
    S3.<X3> = F3x.quotient(D) # ring in 3/DN

    Sq.<x> = PolynomialRing(Integers(q)) # ring modulo q

    # Create the NTRU polynomials
    while True:
        f, g, f3, fq, h = gen_keys(N, q, d, Zx, S3, Sq, S2, R)
        if fq is not None: # if an f invertible in q is found
            break
    msg = gen_msg(N, d, Zx)
    cipher, r = encrypt(N, d, msg[1], h, Zx, Sq, R)     # the ciphertext and the ephemeral key
    decr_m = decrypt(cipher, f, f3, msg[1], Zx, S3, Sq, R)    # we check if the decryption is correct

    # Compute vectors a and a0
    known_msg = msg[0][:k]
    a0 = calculate_c_minus_m(cipher, known_msg)
    a = []
    for i in range(k):
        a.append(get_coeffs_of_right_side(N, h, i))

    return msg, r, cipher, a, a0, known_msg

In [17]:
variant = 3
counter = 0
experiments = 10
for experiment in range(experiments):
    print(experiment, "----------------------------------------------------------------------")
    found = False
    N,q,d,k,N1,N2 = ntruhps(variant)
    # N = 53; q = 2^10; d = 1; N1 = 10; N2 = 4; k=25

    print("k/N * 100:",(k/N).n() * 100)
    print("N - k :", N-k)
    msg, r, cipher, a, a0, known_msg = create_NTRU(N, q, d, k)
    for j in range(1): # change to range(-2,3) to test more N1 values
        tempN1 = N1 + j
        for l in range(1): # change to range(-2,3) to test more N2 values
            temp = N2 + l
            tempN2 = ceil(q^temp)
            our_set = {-1,0,1}
            print("N1, N2 = ", tempN1, temp)

            filename = str(N)+"_"+str(k)
            B = create_fplll_file(filename, N, a, a0, tempN1, tempN2, q, k)
            print("Original basis dimension:", B.dimensions())

            current_path = Path.cwd()
            matrix_path = str(current_path) + "/matrices/" + filename
            command = flatter_path + " < "+ matrix_path

            output = subprocess.run(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True).stdout
            rows = output.strip("[] \n").split('\n')
            matrix_data = [list(map(int, re.findall(r'-?\d+', row))) for row in rows]
            A = matrix(matrix_data)

            column= A.column(N)[:N+1] # we get the N+1th column of the reduced matrix and we check the first N+1 elements
            solutions = []
            homoge = []
            for count, element in enumerate(column): # if and element is a multiple of N1 we add the vector of that row as a possible solution
                quotient = element/tempN1
                row = A[count][:N] # get the first N elements of the row of that element
                if element%tempN1 == 0 and element != 0: # we found a multiple of N1
                    if abs(gcd(row)) == abs(quotient): # if we can divide the elements with the quotient
                        row_divided = row/quotient
                        solutions.append([count, row_divided]) # add it to the possible solutions
                if int(element) == 0:
                    homoge.append(row)

            for solution in solutions:
                if try_row(solution[1], homoge):
                    found = True
                    counter += 1
                    break # no need to see the other solutions
            if found:
                # print("N1, N2: ",tempN1, temp)
                break # break from N2 loop
        if found:
            break # break from N1 loop
print("=========================================================")
print("Successfull exp: ", counter)

0 ----------------------------------------------------------------------
k/N * 100: 92.5700365408039
N - k : 61
N1, N2 =  7 21
Original basis dimension: (1582, 1582)
1 ----------------------------------------------------------------------
k/N * 100: 92.5700365408039
N - k : 61
N1, N2 =  7 21
Original basis dimension: (1582, 1582)


'Correct r found: ' True

2 ----------------------------------------------------------------------
k/N * 100: 92.5700365408039
N - k : 61
N1, N2 =  7 21
Original basis dimension: (1582, 1582)


'Correct r found: ' True

3 ----------------------------------------------------------------------
k/N * 100: 92.5700365408039
N - k : 61
N1, N2 =  7 21
Original basis dimension: (1582, 1582)
4 ----------------------------------------------------------------------
k/N * 100: 92.5700365408039
N - k : 61
N1, N2 =  7 21
Original basis dimension: (1582, 1582)


...

5 ----------------------------------------------------------------------
k/N * 100: 92.5700365408039
N - k : 61
N1, N2 =  7 21
Original basis dimension: (1582, 1582)
6 ----------------------------------------------------------------------
k/N * 100: 92.5700365408039
N - k : 61
N1, N2 =  7 21
Original basis dimension: (1582, 1582)


'Correct r found: ' True

7 ----------------------------------------------------------------------
k/N * 100: 92.5700365408039
N - k : 61
N1, N2 =  7 21
Original basis dimension: (1582, 1582)
8 ----------------------------------------------------------------------
k/N * 100: 92.5700365408039
N - k : 61
N1, N2 =  7 21
Original basis dimension: (1582, 1582)


'Correct r found: ' True

9 ----------------------------------------------------------------------
k/N * 100: 92.5700365408039
N - k : 61
N1, N2 =  7 21
Original basis dimension: (1582, 1582)
Successfull exp:  5
